<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Evaluate without a full answer key

**[Evaluate without a full answer key](https://learning.nextia-ai.com/courses/anomaly-detection/m07/evaluate-without-a-full-answer-key/)** · Anomaly Detection: Find What Is Unusual · Module 7, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** estimate a detector's precision when nobody knows all the answers: from planted anomalies, from a random sample of alerts that people review, and from the agreement of two detectors. For each one, you see what it tells you and what it hides.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. |
| **You should know** | Precision at the budget, from [The alert budget](https://learning.nextia-ai.com/courses/anomaly-detection/m07/the-alert-budget/); kNN distance and Isolation Forest, from Module 4. |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m07/evaluate-without-a-full-answer-key/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M07-L03-evaluate-without-a-full-answer-key/without-a-full-answer-key-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A tiny example: 20 reviewed alerts

A reviewer checked 20 alerts, and 11 were real frauds. The share is 0.55. How sure are we? The **95% interval** is a range that holds the true share in about 95 of 100 samples like this one.

> **Check.** You should see an estimate of 0.55 with an interval from 0.33 to 0.77. With 200 reviewed alerts, the margin is about 0.07.

In [ ]:
real, reviewed = 11, 20                      # a reviewer checked 20 alerts, and 11 were real frauds
estimate = real / reviewed
margin = 1.96 * np.sqrt(estimate * (1 - estimate) / reviewed)   # the 95% interval of a share
print(f"estimate {estimate:.2f}, 95% interval {estimate - margin:.2f} to {estimate + margin:.2f}")
for n in (20, 50, 200, 800):
    print(f"{n:>4} reviewed alerts: plus or minus {1.96 * np.sqrt(0.5 * 0.5 / n):.3f} around 0.50")

A small sample gives a wide interval. Four times more reviews make the margin only half as wide.

## 2. Two detectors on Larkfield's payments

Isolation Forest and kNN distance on the behaviour features, as in Module 4. Each sends its top 50 a day.

> **Check.** You should see 4,550 alerts for each detector.

In [ ]:
import compare, features
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
behaviour = features.behaviour(payments)
scaler = StandardScaler().fit(behaviour.loc[reference.index])
X_ref = scaler.transform(behaviour.loc[reference.index])
X_eval = scaler.transform(behaviour.loc[evaluation.index])
forest = IsolationForest(n_estimators=200, random_state=0, n_jobs=-1).fit(X_ref)
iforest = -forest.score_samples(X_eval)
sample = X_ref[np.random.default_rng(0).choice(len(X_ref), 20_000, replace=False)]   # kNN, as in Module 4
dist, _ = NearestNeighbors(n_neighbors=10).fit(sample).kneighbors(X_eval)
knn = dist[:, -1]
alerts_if = compare.daily_alerts(evaluation, iforest, 50)
alerts_knn = compare.daily_alerts(evaluation, knn, 50)
print(f"Isolation Forest: {alerts_if.reviewed.sum():,} alerts; kNN: {alerts_knn.reviewed.sum():,} alerts (50 a day, 91 days)")

## 3. Way 1: planted anomalies

The course data has a key of planted frauds, so precision and recall are exact. But the key only knows the kinds of fraud that we planted.

> **Check.** You should see a precision of 0.5 and a recall per kind: account takeover 0.443, card testing 0.718, stolen card 0.905.

In [ ]:
print("precision at 50 from the planted key:", compare.score_payments(evaluation, iforest, "Isolation Forest")["precision at budget"])
compare.recall_by_kind(evaluation, iforest)

A new kind of fraud that nobody planted is not in this number at all.

## 4. Way 2: a random sample that people review

This cell **plays the reviewer**: it uses the key only to answer "real or not" for the 200 alerts it picks. In real work, a person answers.

> **Check.** You should see 95 of 200 real: an estimate of 0.475, with an interval from 0.406 to 0.544. The true precision is 0.500.

In [ ]:
# Pretend the key does not exist: a reviewer checks 200 alerts chosen at random.
planted_ids = set(compare.planted_payments().payment_id)        # the reveal: only to play the reviewer
reviewed_alerts = sorted(alerts_if.payment_id[alerts_if.reviewed])
is_real = np.array([p in planted_ids for p in reviewed_alerts])
rng = np.random.default_rng(7)
picked = rng.choice(len(reviewed_alerts), 200, replace=False)
estimate = is_real[picked].mean()
margin = 1.96 * np.sqrt(estimate * (1 - estimate) / 200)
print(f"{is_real[picked].sum()} of 200 are real: estimate {estimate:.3f}, 95% interval {estimate - margin:.3f} to {estimate + margin:.3f}")
print(f"the true precision of all {len(reviewed_alerts):,} alerts: {is_real.mean():.3f}")

One sample is one draw. Run 1,000 samples of 200 to see how much the estimate moves.

> **Check.** The estimates go from 0.395 to 0.610, and the interval holds the truth in 94.1% of the samples.

In [ ]:
rng = np.random.default_rng(2025)
draws = np.array([is_real[rng.choice(len(reviewed_alerts), 200, replace=False)].mean() for _ in range(1000)])
margins = 1.96 * np.sqrt(draws * (1 - draws) / 200)
holds = (draws - margins <= is_real.mean()) & (is_real.mean() <= draws + margins)
fig, ax = plt.subplots()
ax.hist(draws, bins=np.arange(0.38, 0.63, 0.01))
ax.axvline(is_real.mean(), color="black", label=f"true precision {is_real.mean():.3f}")
ax.set_xlabel("estimated precision from 200 reviewed alerts"); ax.set_ylabel("samples"); ax.legend()
plt.show()
print(f"1,000 samples: from {draws.min():.3f} to {draws.max():.3f}; the interval holds the truth in {holds.mean():.1%}")

## 5. Way 3: agreement between two detectors

Where both detectors alert, the alert is more often real. Agreement needs no key at all, but it cannot see what both detectors miss.

> **Check.** You should see an agreement of 0.53, a precision of 0.653 where both alert, and 0.17 where only kNN alerts.

In [ ]:
a = set(alerts_if.payment_id[alerts_if.reviewed])
k = set(alerts_knn.payment_id[alerts_knn.reviewed])
groups = {"both": a & k, "only Isolation Forest": a - k, "only kNN": k - a}
print(f"agreement (Jaccard): {len(a & k) / len(a | k):.2f}")
pd.DataFrame.from_dict({name: {"alerts": len(ids), "real (from the key)": sum(p in planted_ids for p in ids),
                               "precision": round(np.mean([p in planted_ids for p in ids]), 3)}
                        for name, ids in groups.items()}, orient="index")

## 6. Check yourself

**Your turn 1.** Repeat the reviewed sample with only 50 alerts: use `np.random.default_rng(7)` again, pick 50 alerts, and write the margin of the 95% interval (the part after "plus or minus"), rounded to 3 decimals, as `margin_50`.

In [ ]:
margin_50 = ...   # your code

In [ ]:
expect('margin_50', margin_50, 0.138, tolerance=0.001)

**Your turn 2.** How many planted payments of the evaluation months are in **neither** detector's alerts? Write the count as `missed_by_both`. Agreement can never show you these.

In [ ]:
missed_by_both = ...   # your code

In [ ]:
expect('missed_by_both', missed_by_both, 303)

## Recap

- Planted anomalies give exact numbers, but only for the kinds you planted.
- A random sample of reviewed alerts estimates precision with an interval: 200 reviews give about plus or minus 0.07.
- Agreement between detectors tells you where to look first, but not what both miss.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Slices and fairness](https://learning.nextia-ai.com/courses/anomaly-detection/m07/slices-and-fairness/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m07/evaluate-without-a-full-answer-key/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.